# 07 — Network statistics for Task A: Long Length of Stay (issues #11, #12, #13, #15)

**The task:** whether a hospital stay will be long, predicted at the end of the admission day. Each node is one labeled prediction (6,990 admissions from 3,850 patients). Base rate of the outcome: about 25%.

This notebook describes the three patient-similarity graphs for this task:

| Graph | Built from | Role |
|---|---|---|
| **safe** | Each node's diagnoses *before* the prediction time | The honest graph we'd actually use |
| **leaky** | The patient's diagnoses over their *whole* record, including the future | A deliberate cheat, used as a control |
| **null** | The safe graph with its edges randomly rewired. Every node keeps its exact degree, and the edge weights are shuffled | A "structure means nothing" control |

It answers four questions:

1. **Shape (#11).** How many connections do nodes have, how clustered is the graph, and which nodes sit in the middle?
2. **Groups (#12).** Do Louvain and Leiden agree on the groups? Do groups differ in their long stay rate?
3. **Alignment with the outcome (#13).** Homophily, and Dirichlet energy of the CLMBR fingerprints.
4. **What leakage actually changes (#15).** Which links the leaky graph adds or removes, how much of each graph is a patient linked to their own other admissions, and whether any single diagnosis code gives the outcome away.

**One null graph is not enough.** Every null number is the mean over **10 differently seeded null graphs**, with the range covering the middle 95% of them.

**Data rule.** This runs locally against the on-device dataset. Only per-graph summary numbers (`results/aggregate/network_stats_task_a.csv`) and figures built from them (`results/aggregate/figures/07_*.png`) are committed. Per-node tables stay in memory.

**One code path.** All computation lives in `ehrgraph.netstats.task_analysis` and all figures in `ehrgraph.viz.network_figures`, shared by both tasks so their numbers are directly comparable (issue #17).

In [ ]:
# Standard bootstrap cell (identical in every notebook -- see notebooks/README.md):
# find the repo root, put it on the import path, and point EHRSHOT_DATA_ROOT
# at the local, on-device copy of the data.
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from ehrgraph.data.loader import open_database
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.graphs.similarity import graph_summary
from ehrgraph.netstats.task_analysis import (
    PRIMARY_VARIANT,
    analyze_task,
    load_task_graphs,
    summary_table,
)
from ehrgraph.viz import network_figures as nf

TASK = "guo_los"
POSITIVE, NEGATIVE = "long stay", "not long stay"
FIG = "07"  # figure-file prefix
results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
nf.apply_style()

## Build the graphs and run the analysis

`load_task_graphs` builds the safe and leaky node vectors (diagnoses only, binary per node), turns each into a mutual-10-NN similarity graph, and makes 10 degree-preserving null graphs from the safe one. It also aligns and centers the CLMBR fingerprints on the same `(patient_id, prediction_time)` key that notebook 01 validated. `analyze_task` then computes every metric in this notebook in one pass.

In [ ]:
t0 = time.time()
db = open_database(str(meds_reader_path))
tg = load_task_graphs(db, assets, TASK)
print(f"{len(tg.nodes):,} nodes, prevalence {tg.prevalence:.3f}; built in {time.time() - t0:.0f}s")

t0 = time.time()
an = analyze_task(tg, graph_variant=PRIMARY_VARIANT)
print(f"analysis: {len(an.rows)} aggregate rows in {time.time() - t0:.0f}s")
pd.DataFrame({name: graph_summary(g) for name, g in tg.graphs.items()}).T

---
# Part 1 — Shape of the graphs (issue #11)

- **Density**: the share of all possible node pairs that are connected.
- **Clustering** and **transitivity**: when two nodes share a neighbor, how often they are also connected (triadic closure, Lecture 3).
- **Degree assortativity**: whether well-connected nodes link to other well-connected nodes.
- **Path length** and **diameter**: hops between nodes in the largest connected piece. The average path is estimated from 500 sampled start nodes, which matched the exact value to 0.001 hops on Task A.

The null keeps every node's degree, so density and degree match the safe graph by construction. Its clustering and path lengths show how much structure exists beyond "who has how many links."

In [ ]:
STRUCT = [
    "density",
    "mean_degree",
    "max_degree",
    "avg_clustering",
    "transitivity",
    "degree_assortativity",
    "lcc_avg_shortest_path_hops",
    "lcc_diameter_hops_lower_bound",
    "n_components",
    "isolated_fraction",
    "largest_cc_fraction",
]
summary_table(an.rows, STRUCT)

### Degree distribution

Mutual k-NN with k = 10 caps every node at 10 links, so there are no hubs by design. The interesting part is the left side: a node with 0 or 1 links is one that nobody else counts among their closest matches.

In [ ]:
fig = nf.degree_figure(tg)
nf.save(fig, figures_dir / f"{FIG}_degree_distribution.png")
plt.show()

### Centrality

Betweenness and closeness use `distance = 1 − similarity`, and are computed exactly with igraph. Closeness and eigenvector centrality are computed inside the largest connected piece only. Eigenvector centrality *localizes* in graphs this modular: nearly all its weight lands on one dense cluster, so it is not a useful ranking of nodes.

Each AUROC below treats centrality alone as a predictor of long stay. 0.5 means no information. It is descriptive only: it uses every node's label.

In [ ]:
fig = nf.centrality_figure(an)
nf.save(fig, figures_dir / f"{FIG}_centrality_distributions.png")
plt.show()
summary_table(
    an.rows,
    [
        f"{m}_centrality_auroc_vs_label"
        for m in ("degree", "betweenness", "closeness", "eigenvector")
    ],
    conditions=("safe", "leaky"),
).round(3)

---
# Part 2 — Groups (issue #12)

Louvain and Leiden both maximize *modularity*: dense inside groups, sparse between them. Their agreement (adjusted Rand index, ARI) runs from 0 (chance) to 1 (identical).

The figure shows each Leiden group's long stay rate, for groups of at least 20 nodes. Smaller groups are noisy, and a tiny group's rate says too much about the few patients in it. **η²** is the share of outcome variation explained by group membership.

In [ ]:
summary_table(
    an.rows,
    [
        "adjusted_rand_index",
        "n_leiden_communities",
        "louvain_modularity",
        "leiden_modularity",
        "leiden_outcome_eta_squared",
        "leiden_outcome_nmi",
        "leiden_outcome_share_in_reported_communities",
    ],
)

In [ ]:
fig = nf.community_outcome_figure(tg, an, POSITIVE)
nf.save(fig, figures_dir / f"{FIG}_community_outcome_rates.png")
plt.show()

### What the graphs look like

The largest connected piece of each graph, drawn with a force-directed layout. Each dot is one node, colored only by outcome: **violet** = long stay, **light gray** = not long stay. Nothing is labeled or identifiable. Homophily shows up as violet dots clumping together; the null panel shows what no clumping looks like.

In [ ]:
fig = nf.layout_figure(tg, POSITIVE, NEGATIVE)
nf.save(
    fig, figures_dir / f"{FIG}_graph_layouts.png", dpi=80
)  # thousands of dots; keeps it < 500 KB
plt.show()

---
# Part 3 — Do the graphs line up with the outcome? (issue #13)

**Class-insensitive homophily** (Lim et al., 2021): for each outcome, how often its nodes link to same-outcome nodes, minus what random wiring would give. Random wiring scores about **0** at any prevalence, and perfect segregation scores **1**. That correction matters because Task B (Hypertension) has a different prevalence (0.14).

**Dirichlet energy** asks the same question of the CLMBR fingerprints: how different a node's fingerprint is from its neighbors', normalized across graphs. **Lower means neighbors already look alike to CLMBR.**

Both use every node's label or fingerprint. They describe the graph, and must never be used to choose a model.

In [ ]:
summary_table(
    an.rows,
    [
        "edge_homophily",
        "class_insensitive_homophily",
        "same_outcome_rate_positive",
        "same_outcome_rate_negative",
        "dirichlet_energy_clmbr",
    ],
)

In [ ]:
fig = nf.homophily_energy_figure(an)
nf.save(fig, figures_dir / f"{FIG}_homophily_dirichlet.png")
plt.show()

---
# Part 4 — What does leakage actually change? (issue #15)

Three checks:

1. **Same-patient links.** A patient with several labeled admissions appears as several nodes, and those nodes tend to link to each other: their histories overlap, and in the leaky graph they are *identical*, since every admission sees the whole record. Those links share an outcome far more often than chance. So homophily is reported both with and without them. The official splits are per patient, so these links never cross train and test.
2. **Which links leakage swaps.** Links in both graphs, only in safe, and only in leaky. If leakage helps, the leaky-only links should join same-outcome nodes more often than the safe-only links they replaced.
3. **Single-code leakage.** For each diagnosis code, the share of positive vs. negative nodes whose vector contains it. In the leaky vectors a huge gap means the future record contains the answer itself.

In [ ]:
summary_table(
    an.rows,
    [
        "same_patient_edge_fraction",
        "same_patient_edge_same_outcome_rate",
        "class_insensitive_homophily",
        "class_insensitive_homophily_excl_same_patient",
        "max_single_code_outcome_gap",
    ],
)

In [ ]:
edge_sets = pd.DataFrame(
    {k: an.edge_sets[k] for k in ("shared_edges", "safe_only_edges", "leaky_only_edges")}
).T
print(f"edge Jaccard (safe vs leaky): {an.edge_sets['edge_jaccard']:.3f}")
edge_sets

In [ ]:
# The ten codes whose presence in the LEAKY vectors best separates positives
# from negatives -- code names and group rates only, no patient-level data.
an.top_leaky_codes.round(3)

In [ ]:
fig = nf.leakage_edges_figure(tg, an)
nf.save(fig, figures_dir / f"{FIG}_leakage_edges.png")
plt.show()

---
## Save the aggregate results

`write_aggregate` checks the schema and refuses patient-identifying columns before anything touches disk.

In [ ]:
out = write_aggregate(an.rows, results_dir / "network_stats_task_a.csv")
saved = pd.read_csv(out)
print(f"wrote {len(saved)} rows -> {out.relative_to(_repo_root)}")
print(sorted(p.name for p in figures_dir.glob(f"{FIG}_*.png")))
saved.groupby("condition")["metric"].count()

## Findings

Numbers first computed Sep 25, 2026 and reproduced exactly on Oct 2 through the shared library (all 169 original values identical). 6,990 admissions, long-stay prevalence 0.253, primary graph configuration. "Null" means the mean of 10 rewired graphs, with the middle-95% range in brackets.

**1. The real graphs are "large worlds" made of tight clusters, and that structure is real.** All three graphs have identical degrees: mean 5.35, capped at 10 by mutual k-NN. The wiring is what differs:

| | safe | leaky | null |
|---|---|---|---|
| transitivity (closed triangles) | 0.498 | 0.660 | 0.001 [0.0008–0.0012] |
| average path in the largest piece (hops) | 9.6 | 11.9 | 5.3 [5.27–5.31] |
| largest piece (share of nodes) | 87.4% | 61.9% | 95.5% |

The rewired graph is a textbook small world with short paths and almost no triangles. The real graphs cluster about 500× more and have paths twice as long.

**2. The leaky graph has the "excess of closed triangles" PROJECT_GUIDE.md Module C flagged as a warning sign.** Its transitivity is 0.66 against 0.50 for safe. It is also more fragmented: 967 pieces against 408. This extends the mechanism from the notebook-05 follow-up. Seeing the future makes patients' vectors fuller and noisier, so fewer pairs choose each other mutually, and the pairs that do tend to sit in tight, closed cliques.

**3. Community structure is strong, stable, and tied to the outcome.** Modularity is 0.94 for safe and 0.98 for leaky, against 0.56 [0.553–0.560] for null. Louvain and Leiden largely agree: ARI 0.71 for safe and 0.82 for leaky, against 0.22 for null. Among Leiden groups of at least 20 patients, long-stay rates run from about 0% to 72% in the safe graph. Group membership explains **η² = 0.121** of the outcome variance for safe and 0.147 for leaky, against 0.007 [0.006–0.010] for null.

**4. Connected patients share outcomes far more than chance (issue #13).** Class-insensitive homophily is **0.209 (safe)** and 0.201 (leaky), against **0.011 [0.009–0.014]** for null. Put concretely, 40% of a long-stay patient's neighbors are also long-stay, against a 25% base rate. The CLMBR fingerprints tell the same story: Dirichlet energy is **0.683 (safe)** and 0.685 (leaky), against **0.953 [0.952–0.955]** for null. Graph neighbors already look alike to CLMBR.

**5. Centrality says nothing about the outcome.** Using any centrality measure as a stand-in predictor gives AUROC 0.47–0.51 across almost all graphs. The one exception is the leaky graph's eigenvector centrality at 0.55, and that measure is unreliable here anyway. Eigenvector centrality *localizes*: even inside the largest piece, over 97% of nodes score about 0 because the leading eigenvector concentrates on one dense cluster, which is expected in graphs this modular. Don't use it as a ranking of patients.

**6. What leakage actually changes (issue #15).**

- **Over a third of all links join two admissions of the same patient:** 37.5% of safe links and 39.6% of leaky links, against 0.1% in the null. Those links share an outcome only 68% of the time, barely above the 62% chance rate for random pairs, because the same person's stays vary in length. Removing them *raises* homophily: 0.209 → **0.237** for safe and 0.201 → **0.242** for leaky. So same-patient links don't inflate Task A's homophily; they dilute it. Task B is the opposite (notebook 10).
- **Leakage swaps out about half the links, and the swap doesn't help.** The safe and leaky edge sets overlap only 37% (Jaccard). The links only the leaky graph has join same-outcome patients **68.5%** of the time, while the links it removed were at **71.7%** (chance: 62%). For long stay, seeing the future replaces links with slightly *worse* ones.
- **No single future diagnosis gives long stay away.** The strongest single code in the leaky vectors, ICD-10 R91.8 (abnormal lung imaging), appears in 50% of long stays against 34% of others, a gap of 0.16. The safe vectors' best is 0.11. Long stay isn't a diagnosis code, so the future record holds no answer key for it.

### What this means for the project

- **For long stay, leakage changes the graph's *shape* but adds no outcome signal.** Homophily matches with or without same-patient links, and the links leakage brings in are slightly worse than the ones it removes. If the leaky graph nonetheless *predicts* better in Project 2, that would point to something other than outcome alignment, and it should be investigated rather than taken at face value.
- **Report homophily both ways.** Same-patient links make up over a third of every real graph. Task A is robust to them, but Task B is not (notebook 10), so every homophily number in the report should come with its without-same-patient twin.
- **Low Dirichlet energy cuts both ways for Project 2.** Graph neighbors already have similar CLMBR fingerprints, so smoothing over them should reduce noise. It may also add little information CLMBR doesn't already have. This is the concrete version of the core question, and a reason to expect small gains rather than large ones.

### Caveats

- One graph configuration: diagnoses only, binary, TF-IDF, mutual k-NN with k = 10. The graph-variant sweep (Part 4, item 12) comes later.
- Homophily, group outcome rates, and centrality AUROC use *every* node's label. That's fine for describing the graph, but none of these numbers may be used to choose a model.
- The average path length is estimated from breadth-first search out of 500 sampled patients (9.559 vs the exact 9.558 on the safe graph). Everything else is exact.

### Done when
- [x] #11: degree distribution, structural statistics, and all four centralities (exact) for safe, leaky, and a 10-seed null ensemble
- [x] #12: Louvain + Leiden, agreement (ARI), modularity, and group outcome rates, each compared against the null range
- [x] #13: class-insensitive homophily (Lim et al. 2021) and normalized Dirichlet energy on centered CLMBR fingerprints
- [x] #15: same-patient links, leakage edge-set comparison, single-code leakage
- [x] Only aggregate output written: `results/aggregate/network_stats_task_a.csv` (193 rows) and 6 figures